# 03 - Evaluate runs and create tables

    This notebook evaluates the trained runs with validation denoising loss and, when feasible, small-sample FID/KID on Imagenette. These metrics are for relative comparisons only and are not comparable to the paper's ImageNet FID-50K.


## Imports and paths

    This cell imports project modules and creates folders for generated evaluation samples, CSV results, and figures.


In [ ]:
import sys, shutil, json, math
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))

from dit.configs import make_dit_config, RUNS
from dit.models import make_model
from dit.diffusion import GaussianDiffusion
from dit.ema import EMA
from dit.train_utils import load_checkpoint, validation_loss, make_loader
from dit.data import LatentTensorDataset
from dit.latent import load_vae, decode_latents_to_images
from dit.sample_utils import save_tensor_images
from dit.metrics import compute_clean_fid, compute_torch_fidelity
from torchvision.utils import save_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = PROJECT_ROOT / "data"
LATENT_DIR = DATA_DIR / "imagenette_latents"
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "latent_imagenette"
SAMPLE_ROOT = PROJECT_ROOT / "samples" / "latent_imagenette_eval"
RESULT_DIR = PROJECT_ROOT / "results"
FIG_DIR = PROJECT_ROOT / "figures"
for d in [SAMPLE_ROOT, RESULT_DIR, FIG_DIR]: d.mkdir(parents=True, exist_ok=True)

## Create a flat real-image reference folder

    FID tools work best with a flat folder of images. This cell creates `data/eval_refs/imagenette_val_flat` from Imagenette validation images using symlinks when possible, otherwise copies.


In [ ]:
real_root = DATA_DIR / "imagenette2-320" / "val"
ref_dir = DATA_DIR / "eval_refs" / "imagenette_val_flat"
ref_dir.mkdir(parents=True, exist_ok=True)
if len(list(ref_dir.glob("*.JPEG"))) + len(list(ref_dir.glob("*.jpg"))) + len(list(ref_dir.glob("*.png"))) == 0:
    idx = 0
    for img_path in real_root.rglob("*"):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        target = ref_dir / f"real_{idx:06d}{img_path.suffix}"
        try:
            target.symlink_to(img_path.resolve())
        except Exception:
            shutil.copy2(img_path, target)
        idx += 1
print("Reference images:", len(list(ref_dir.iterdir())), ref_dir)


## Load validation latents and VAE

    This cell prepares validation-loss evaluation and latent decoding for sample generation.


In [ ]:
val_latents = LatentTensorDataset(LATENT_DIR / "val")
val_loader = make_loader(val_latents, batch_size=32, shuffle=False, num_workers=4)
vae = load_vae(device)
diffusion = GaussianDiffusion(1000, device=device)


## Evaluation helpers

    This cell defines helpers that load a checkpoint, generate evaluation samples if needed, compute validation loss, and compute FID/KID. Missing checkpoints are marked explicitly rather than creating blank CSV rows.


In [ ]:
NUM_EVAL_SAMPLES = 1000  # increase to 2000 or 5000 if time allows
DDIM_STEPS = 100
CFG_SCALE = 1.5
FORCE_REGENERATE = False

def load_model_for_run(run_id):
    ckpt_path = CKPT_ROOT / run_id / "latest.pt"
    if not ckpt_path.exists():
        return None, None, None, "missing_checkpoint"
    ckpt = torch.load(ckpt_path, map_location="cpu")
    allowed = ["model_name", "patch_size", "input_size", "in_channels", "num_classes", "class_dropout_prob", "learn_sigma", "mlp_ratio", "conditioning"]
    cfg = make_dit_config(**{k: v for k, v in ckpt["dit_config"].items() if k in allowed})
    model = make_model(cfg).to(device)
    ema = EMA(model, decay=ckpt.get("train_config", {}).get("ema_decay", 0.999))
    load_checkpoint(ckpt_path, model, ema=ema, device=device)
    if ckpt.get("ema") is not None:
        ema.apply_to(model)
    model.eval()
    return model, cfg, ckpt, "ok"

@torch.no_grad()
def generate_eval_samples(model, cfg, out_dir, num_samples=NUM_EVAL_SAMPLES):
    out_dir = Path(out_dir)
    existing = list(out_dir.glob("*.png"))
    if len(existing) >= num_samples and not FORCE_REGENERATE:
        return
    if out_dir.exists() and FORCE_REGENERATE:
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    batch_size = 32 if device.type == "cuda" else 4
    written = 0
    pbar = tqdm(total=num_samples, desc=f"Generating {out_dir.name}")
    while written < num_samples:
        n = min(batch_size, num_samples - written)
        labels = torch.arange(n, device=device) % cfg.num_classes
        latents = diffusion.ddim_sample_loop(model, (n, cfg.in_channels, cfg.input_size, cfg.input_size), labels, steps=DDIM_STEPS, cfg_scale=CFG_SCALE, device=device, progress=False)
        images = decode_latents_to_images(vae, latents)
        save_tensor_images(images.cpu(), out_dir, start_index=written)
        written += n
        pbar.update(n)
    pbar.close()

def evaluate_run(run_spec):
    run_id = f"{run_spec['model_name']}-{run_spec['patch_size']}"
    model, cfg, ckpt, status = load_model_for_run(run_id)
    row = {
        "run_id": run_id,
        "model": run_spec["model_name"],
        "patch_size": run_spec["patch_size"],
        "tokens": None,
        "checkpoint_step": None,
        "cfg_scale": CFG_SCALE,
        "ddim_steps": DDIM_STEPS,
        "num_generated": NUM_EVAL_SAMPLES,
        "fid": float("nan"),
        "kid_mean": float("nan"),
        "val_loss": float("nan"),
        "t_000_100_loss": float("nan"),
        "t_100_300_loss": float("nan"),
        "t_300_600_loss": float("nan"),
        "t_600_1000_loss": float("nan"),
        "sample_dir": "",
        "status": status,
        "fid_status": "not_run",
    "kid_status": "not_run",
    }

    if status != "ok":
        return row
    row["tokens"] = cfg.num_tokens
    row["checkpoint_step"] = ckpt.get("step")
    vals = validation_loss(model, diffusion, val_loader, device, max_batches=50)
    row.update(vals)
    fake_dir = SAMPLE_ROOT / run_id / f"step_{ckpt['step']:07d}_cfg_{CFG_SCALE}_ddim_{DDIM_STEPS}_n{NUM_EVAL_SAMPLES}"
    generate_eval_samples(model, cfg, fake_dir, NUM_EVAL_SAMPLES)
    row["sample_dir"] = str(fake_dir)
    
    # try:
    #     row["fid"] = compute_clean_fid(ref_dir, fake_dir, num_workers=0)
    #     row["status"] = "ok"
    # except Exception as exc:
    #     row["status"] = f"fid_error: {type(exc).__name__}: {exc}"
    # try:
    #     tf = compute_torch_fidelity(ref_dir, fake_dir, kid=True, fid=False)
    #     for k, v in tf.items():
    #         if "kid" in k.lower() and "mean" in k.lower():
    #             row["kid_mean"] = v
    # except Exception as exc:
    #     if row["status"] == "ok":
    #         row["status"] = f"kid_error: {type(exc).__name__}: {exc}"

    try:
        row["fid"] = compute_clean_fid(ref_dir, fake_dir, num_workers=0)
        row["fid_status"] = "ok"
        row["status"] = "ok"
    except Exception as exc:
        row["fid_status"] = f"fid_error: {type(exc).__name__}: {exc}"
        row["status"] = "fid_error"

    try:
        tf = compute_torch_fidelity(ref_dir, fake_dir, kid=True, fid=False)
        for k, v in tf.items():
            if "kid" in k.lower() and "mean" in k.lower():
                row["kid_mean"] = v
        row["kid_status"] = "ok"
    except Exception as exc:
        row["kid_status"] = f"kid_error: {type(exc).__name__}: {exc}"

    # Important: KID failure should not invalidate the row if FID or validation loss exists.
    if row["fid_status"] == "ok":
        row["status"] = "ok_fid_only" if row["kid_status"] != "ok" else "ok"
    elif not math.isnan(row["val_loss"]):
        row["status"] = "ok_val_loss_only"

    return row


## Run evaluation for the configs

    This cell evaluates each planned checkpoint. Every CSV row is either a real measured result or explicitly marked with a status such as `missing_checkpoint`.


In [ ]:
rows = []
for run_spec in RUNS:
    rows.append(evaluate_run(run_spec))
results = pd.DataFrame(rows)
out_csv = RESULT_DIR / "metrics.csv"
results.to_csv(out_csv, index=False)
display(results)
print("Saved:", out_csv)


## Plot validation loss and FID against token count

    This cell creates the main figures for small-scale DiT scaling. 

In [ ]:
valid_loss = results[results["val_loss"].notna()].copy()
valid_fid = results[results["fid"].notna()].copy()

if len(valid_loss):
    plt.figure(figsize=(6, 4))
    plt.scatter(valid_loss["tokens"], valid_loss["val_loss"], s=80)
    for _, r in valid_loss.iterrows():
        plt.text(r["tokens"], r["val_loss"], r["run_id"])
    plt.xlabel("Latent token count")
    plt.ylabel("Validation denoising loss")
    plt.title("DiT scaling: tokens vs validation loss")
    plt.grid(True, alpha=0.3)
    out = FIG_DIR / "tokens_vs_val_loss.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
else:
    print("No rows with validation loss available.")

if len(valid_fid):
    plt.figure(figsize=(6, 4))
    plt.scatter(valid_fid["tokens"], valid_fid["fid"], s=80)
    for _, r in valid_fid.iterrows():
        plt.text(r["tokens"], r["fid"], r["run_id"])
    plt.xlabel("Latent token count")
    plt.ylabel(f"FID-{NUM_EVAL_SAMPLES}")
    plt.title("DiT scaling: tokens vs FID")
    plt.grid(True, alpha=0.3)
    out = FIG_DIR / "tokens_vs_fid.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
else:
    print("No rows with FID available.")